# Official HEC-HMS Guide Mirror: Meteorologic Methods

Official guides:

- https://www.hec.usace.army.mil/confluence/hmsdocs/hmsguides/meteorologic-methods-in-hms
- https://www.hec.usace.army.mil/confluence/hmsdocs/hmsguides/gridded-boundary-condition-data

This notebook maps the official meteorologic-method topics to hms-commander APIs that already exist: sample met model inspection, gage metadata, gridded precipitation detection, Atlas 14 depth-duration preparation, frequency storm hyetographs, and SCS temporal patterns.

In [1]:
from pathlib import Path
import logging

import pandas as pd

logging.disable(logging.CRITICAL)


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "hms_commander").is_dir() and (candidate / "examples").is_dir():
            return candidate
    return start


REPO_ROOT = find_repo_root()
WORK_ROOT = REPO_ROOT / "examples" / "working" / "clb238_guides"
WORK_ROOT.mkdir(parents=True, exist_ok=True)
HMS_VERSION = "4.13"


from hms_commander import HmsExamples, HmsPrj

available_versions = HmsExamples.list_versions()
if HMS_VERSION not in available_versions:
    HMS_VERSION = available_versions[0]
HMS_EXE = HmsExamples.get_hms_exe(HMS_VERSION)


def init_sample_project(project_name, notebook_key):
    project_path = HmsExamples.extract_project(
        project_name,
        version=HMS_VERSION,
        output_path=WORK_ROOT / notebook_key,
        overwrite=True,
    )
    project = HmsPrj()
    project.initialize(project_path, hms_exe_path=HMS_EXE)
    return project, project_path

In [2]:
from hms_commander import Atlas14Storm, FrequencyStorm, HmsGage, HmsMet, ScsTypeStorm

castro, castro_path = init_sample_project("castro", "23_meteorologic_methods")
tenk, tenk_path = init_sample_project("tenk", "23_meteorologic_methods")

met_inventory = pd.concat([
    castro.met_df.assign(sample_project="castro"),
    tenk.met_df.assign(sample_project="tenk"),
], ignore_index=True)
met_inventory[["sample_project", "name", "precip_method", "et_method", "snowmelt_method", "num_subbasin_assignments"]]

,sample_project,name,precip_method,et_method,snowmelt_method,num_subbasin_assignments
0,castro,GageWts,Weighted Gages,No Evapotranspiration,None,4
1,tenk,Stage3-HRAP,Gridded Precipitation,No Evapotranspiration,None,4


In [3]:
gage_file = next(castro_path.glob("*.gage"))
gage_summary = HmsGage.get_gages(gage_file)[["name", "type", "description"]]
project_gage_refs = castro.gage_df[["name", "gage_type", "has_dss_reference"]].copy()
project_gage_refs["dss_path_available"] = castro.gage_df["dss_pathname"].astype(str).str.len() > 0

pd.concat(
    [
        gage_summary.assign(source="gage_file"),
        project_gage_refs.rename(columns={"gage_type": "type"}).assign(description="", source="project_index"),
    ],
    ignore_index=True,
    sort=False,
)[["source", "name", "type", "description", "has_dss_reference", "dss_path_available"]]

,source,name,type,description,has_dss_reference,dss_path_available
0,gage_file,Fire Dept.,Precipitation,Recording precipitation gage,NaN,NaN
1,gage_file,Out,Precipitation,Recording gage at outlet,NaN,NaN
2,project_index,Fire Dept.,Precipitation,,True,True
3,project_index,Out,Flow,,True,True


In [4]:
gridded_met = tenk.met_df.loc[tenk.met_df["precip_method"].str.contains("Gridded", case=False, na=False)]
assert not gridded_met.empty

gridded_assets = pd.DataFrame([
    {"asset_type": "grid_definition", "file": path.name, "exists": path.exists()}
    for path in sorted(tenk_path.glob("*.grid"))
])
assert not gridded_assets.empty
gridded_assets

,asset_type,file,exists
0,grid_definition,tenk.grid,True


In [5]:
durations = Atlas14Storm.FREQUENCY_STORM_DURATIONS_MIN
example_depth_table = pd.DataFrame({
    "duration_minutes": durations,
    100: [0.55, 1.15, 1.90, 2.80, 3.90, 4.60, 6.10, 8.40],
})
frequency_depths = Atlas14Storm.build_frequency_storm_depths(example_depth_table, ari_years=100)
frequency_hyetograph = FrequencyStorm.generate_from_ddf(
    frequency_depths,
    durations=durations,
    peak_position_pct=67.0,
    time_interval_min=15,
)
scs_type_ii = ScsTypeStorm.generate_hyetograph(8.40, scs_type="II", time_interval_min=60)

assert abs(frequency_hyetograph["cumulative_depth"].iloc[-1] - 8.40) < 1e-9
assert abs(scs_type_ii["cumulative_depth"].iloc[-1] - 8.40) < 1e-9

pd.DataFrame([
    {
        "storm": "Frequency storm",
        "rows": len(frequency_hyetograph),
        "duration_hours": frequency_hyetograph["hour"].max(),
        "total_depth_in": frequency_hyetograph["cumulative_depth"].iloc[-1],
    },
    {
        "storm": "SCS Type II",
        "rows": len(scs_type_ii),
        "duration_hours": scs_type_ii["hour"].max(),
        "total_depth_in": scs_type_ii["cumulative_depth"].iloc[-1],
    },
])

,storm,rows,duration_hours,total_depth_in
0,Frequency storm,97,24.25,8.4
1,SCS Type II,25,25.00,8.4


## Coverage Notes

This starter mirror covers meteorologic inventory, historical gage references, gridded precipitation assets, hypothetical frequency storms, and temporal pattern generation. Higher-level authoring for inverse-distance, gage-weight, gridded-met, and historical time-series workflows is tracked separately in CLB-288.